# LC 1143 — Longest Common Subsequence
**Day 40 | 2D Dynamic Programming | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> When two characters match, the LCS
grows by 1 from the diagonal (both strings advance). When they
don't match, we take the best result from skipping either the
current character in text1 or text2. The table encodes all
possible alignments simultaneously.
</div>

## Official Problem Statement

Given two strings `text1` and `text2`, return the length of their
**longest common subsequence**. If there is no common subsequence,
return `0`.

A **subsequence** of a string is a new string generated from the
original string with some characters (can be none) deleted without
changing the relative order of the remaining characters.

A **common subsequence** of two strings is a subsequence that is
common to both strings.

**Constraints:**
- `1 <= text1.length, text2.length <= 1000`
- `text1` and `text2` consist of only lowercase English characters.

## What This Is Actually Asking

Find the longest sequence of characters that appears in both strings
in the same relative order, but not necessarily contiguously.
You can delete characters from either string, but you cannot
rearrange them.
The answer is the length of that longest shared sequence.
For example, `abcde` and `ace` share the subsequence `ace`
which has length 3.

## Walk Through an Example by Hand

**Input:** `text1="abcde"`, `text2="ace"`

Build a `(len(text1)+1) x (len(text2)+1)` table.
Row 0 and col 0 are all 0s (empty string base case).

```
      ""  a  c  e
  ""  [0, 0, 0, 0]
   a  [0, ?, ?, ?]
   b  [0, ?, ?, ?]
   c  [0, ?, ?, ?]
   d  [0, ?, ?, ?]
   e  [0, ?, ?, ?]
```

Fill row by row:
- i=1 ('a'), j=1 ('a'): match → dp[0][0]+1 = **1**
- i=1 ('a'), j=2 ('c'): no match → max(dp[0][2], dp[1][1]) = max(0,1) = **1**
- i=1 ('a'), j=3 ('e'): no match → max(dp[0][3], dp[1][2]) = max(0,1) = **1**
- i=2 ('b'), j=1 ('a'): no match → max(dp[1][1], dp[2][0]) = max(1,0) = **1**
- i=3 ('c'), j=2 ('c'): match → dp[2][1]+1 = 1+1 = **2**
- i=5 ('e'), j=3 ('e'): match → dp[4][2]+1 = 2+1 = **3** ← answer

## The Picture

`text1="abcde"`, `text2="ace"` — completed dp table:

```
      ""   a    c    e
  "" [ 0,  0,   0,   0 ]
   a [ 0,  1,   1,   1 ]
   b [ 0,  1,   1,   1 ]
   c [ 0,  1,   2,   2 ]
   d [ 0,  1,   2,   2 ]
   e [ 0,  1,   2,  [3] ]  <-- answer

  Match:    dp[i][j] = dp[i-1][j-1] + 1  (diagonal + 1)
  No match: dp[i][j] = max(dp[i-1][j], dp[i][j-1])
```

## When To Use This Pattern

- When comparing **two sequences** character by character,
  think **2D DP where axes represent positions in each string**.
- When a problem involves **subsequences** (not substrings),
  think **LCS or its variants**.
- When you need the **edit distance** between strings,
  think **LCS-style table** (Levenshtein is a close cousin).
- When matching needs to respect **order but allow gaps**,
  think **LCS over sliding-window approaches**.
- When the problem mentions diff utilities or version control,
  think **LCS** (it is literally how `diff` works).

## The Approach

Create a 2D table `dp` of size `(m+1) x (n+1)` where `m` and `n`
are the lengths of the two strings, initialized to 0. Iterate
over every pair of characters: if they match, extend the common
subsequence found so far from the diagonal (`dp[i-1][j-1] + 1`);
if they don't match, inherit the best result from skipping one
character in either string (`max(dp[i-1][j], dp[i][j-1])`).
The final answer is at `dp[m][n]`.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    cases = [
        # (text1, text2, expected)
        ("abcde", "ace", 3),
        ("abc", "abc", 3),
        ("abc", "def", 0),
        ("", "abc", 0),        # edge: empty string
        ("abc", "", 0),        # edge: empty string
        ("a", "a", 1),         # edge: single char match
        ("a", "b", 0),         # edge: single char no match
        ("bsbininm", "jmjkbkjkv", 1),
        ("oxcpqrsvwf", "shmtulqrypy", 2),
    ]
    passed = 0
    for text1, text2, expected in cases:
        result = func(text1, text2)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        t1_disp = text1[:10] + "..." if len(text1) > 10 else text1
        t2_disp = text2[:10] + "..." if len(text2) > 10 else text2
        print(
            f"{status} | '{t1_disp}' vs '{t2_disp}' "
            f"| expected={expected}, got={result}"
        )
    print(f"\n{passed}/{len(cases)} tests passed.")

In [ ]:
def longest_common_subsequence(text1: str, text2: str) -> int:
    """
    Find the length of the longest common subsequence
    of text1 and text2.

    Args:
        text1: first input string
        text2: second input string

    Returns:
        Length of the LCS (int), or 0 if none exists.

    Strategy:
        - Build dp table of size (m+1) x (n+1), init to 0
        - If text1[i-1] == text2[j-1]:
              dp[i][j] = dp[i-1][j-1] + 1
          Else:
              dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        - return dp[m][n]

    Time:  O(m * n)
    Space: O(m * n) — reducible to O(n) with two rows
    """
    m, n = len(text1), len(text2)
    print(f"[DEBUG] text1='{text1}' (len={m})")
    print(f"[DEBUG] text2='{text2}' (len={n})")

    # Build the DP table
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    print(f"[DEBUG] dp table initialized ({m+1} x {n+1})")

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i-1] == text2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                print(
                    f"[DEBUG] Match '{text1[i-1]}' at "
                    f"i={i},j={j} → dp={dp[i][j]}"
                )
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    print(f"[DEBUG] LCS length = dp[{m}][{n}] = {dp[m][n]}")

    pass  # replace with: return dp[m][n]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(longest_common_subsequence)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (recursion) | O(2^(m+n)) | O(m+n) stack | All subsequence combos |
| Memoization (top-down) | O(m * n) | O(m * n) | Cached recursion |
| Tabulation 2D (optimal) | O(m * n) | O(m * n) | Full dp table |
| Tabulation 2-row | O(m * n) | O(n) | Only keep 2 rows |

## Real World Connection

LCS is the algorithm behind `git diff` and code review tools —
finding common lines between two file versions to highlight only
what changed. At Citi, reconciliation systems use LCS-style
logic to align transaction sequences from two counterparties
and identify mismatches. In AWS data engineering, schema evolution
detection compares column sequences across versions to find
compatible mappings. LCS also powers DNA sequence alignment tools,
which follow the same 2D DP logic used in large-scale bioinformatics
pipelines on AWS.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra